## Q3. Collage (55 marks)

### Instructions

In this question, you are asked to write functions towards creating collages from a given collection of photos. For simplicity, the resulting collage is always:
* In a square shape
* With boundaries (both inner and outer) that have the following properties:
    * Grey (with rgb`(200,200,200)`)
    * All boundaries have the same width
* with each constituent photo cropped to:
    * Centre to the middle (as much as possible)
    * The same width (as much as possible)
    * The same height (as much as possible)

Materials provided:
* Some example images in the data folder for the example usage of the functions you need to implement
* [`image_util.py`](image_util.py) in the `src` folder for loading and displaying images
    * Please make use of the appropriate functions from [`image_util.py`](image_util.py) in the `src` folder to help you load the photos as 3d `np.ndarray`, and see if the functions you have implemented return the same results as the given example usages

---

### Importing Modules and Libraries for Q3

In [2]:
import time
import numpy as np

from image_util import show
from image_util import load

---

1. (Warm-up, 3 marks) Write the function definition for the function `create_empty_square_img()` below to create a square image in grey colour. The function takes a positive integer (`size`) to represent the height (and the width) of the resulting image. The function returns a square image (3d `np.ndarray` with dtype `np.uint8`) with the shape `(size, size, 3)` filled only with the value `200`. 

    Hint: you may want to make use of [`full()`](https://numpy.org/doc/stable/reference/generated/numpy.full.html) from `numpy`
    
    Note:
    * Loops should not be needed (including related tools like list comprehension)
    * The image given as an argument should not be changed

    Example usage (suppose `image_util` has been imported in previous lines of code):

<img src="../figs/demo_1.png" width=600/>

In [3]:
def create_empty_square_img(size):
    '''
    DESCRIPTION:
        this function returns a grey square image with the height and width
    ----------
    PARAMETERS:
        size: a positive integer to represent the height (and width) of the resulting image
    RETURNS:
        empty_img: a 3d np.ndarray with dtype np.uint8 with shape (size, size, 3) filled with the value 200
    '''
    empty_square = np.full((size,size,3), 200, dtype=np.uint8)
    
    return empty_square

2. (10 marks) Write the function definition for the function `crop()` below to create an image with the middle part of the given image. The function takes an image (3d `np.ndarray` with dtype `np.uint8`) and two positive integers (`h` and `w`) to represent the height and the width of the image to copy. The function returns a new image (3d `np.ndarray` with dtype `np.uint8`) created by copying the middle part of the given image. If the original 3d `np.ndarray` has the shape `(m, n, 3)`, the return 3d `np.ndarray` has the shape `(h, w, 3)`. Assume the given `h` is never greater than `m` and `w` is never greater than `n`.

    Additional Requirement:
    * In the function definition, check if the given arguments satisfy the assumptions and pre-conditions stated above. Raise appropriate types of exceptions if the conditions are not satisfied. 

    Note: 
    * Sometimes it is not possible to have the cropped image exactly in the middle. For example, if the image has a height of 10 and a width of 8, and we want the resulting image to have a height of 4 and a width of 5. While for height there is no issue (keeping 4, 5, 6, 7th rows), in order to have the image to be exactly in the middle, one needs to select the "2.5", "3.5", "4.5", "5.5", "6.5"th columns which are difficult to implement. Instead, the resulting image can be created from the 2-6th columns or 3-7th columns
    * Loops should not be needed (including related tools like list comprehension)
    * The image given as an argument should not be changed

    Example usage (suppose the image `CBG.jpg` in the data folder has been loaded using an appropriate function from `image_util` to form the variable `cbg`, which is a 3d `np.ndarray` representing the image):

<img src="../figs/demo_2.png" width=600/>

In [43]:
def crop(img, h, w):
    '''
    DESCRIPTION:
        this function returns the given image, but cropped  with specificed height (h) and width (w)
    ----------
    PARAMETERS:
        img: a 3d `np.ndarray` with dtype `np.uint8`
        h: a positive integer to represent the height of the cropped image
        w: a positive integer to represent the width of the cropped image
    RETURNS:
        cropped_img: a 3d `np.ndarray` with dtype `np.uint8` created by copying the middle part of the given image
    '''
    num_rows, num_columns, num_dimensions = img.shape   

    if h > num_rows or w > num_columns:
        raise ValueError ('desired height and width of cropped image are larger than height and width of given image')
    if h <= 0 or w <= 0:
        raise ValueError('desired height and width of cropped image must be greater than 0')
    
    start_height = (num_rows - h) // 2
    start_width = (num_columns - w) // 2
    end_height = num_rows - (start_height)
    end_width =  num_columns - (start_width)
    
    cropped_img = img[start_height:(end_height), start_width:(end_width)]

    return cropped_img

3. (11 marks) Write the function definition for the function `crop_w_loop()` for which it takes the same arguments and returns the same value as `crop()`, but it iterates over each pixel instead of using vectorised operations selecting the part of the image to copy. 

    Call both `crop()` and `crop_w_loop()` using the same image `lse.jpg` from the data folder and crop the size to $600 \times 1000$. Time the execution time (e.g. by using the module `time`, `datetime` or [`timeit`](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit)) for each function. How is the execution time different?

    **The execution time differs largely: crop() executes faster than crop_w_loop(), as crop_w_loop() is almost a second, whereas crop() is less than 1 millisecond. Therefore, we can conclude that using vectorised operations is more efficient than iteration over each pixel.**
    
    Note: 
    * You can use loops in this part
    * The image given as an argument should not be changed

In [44]:
def crop_w_loop(img, h, w):
    '''
    DESCRIPTION:
        this function returns the given image, but cropped  with specificed height (h) and width (w)
    ----------
    PARAMETERS:
        img: a 3d `np.ndarray` with dtype `np.uint8`
        h: a positive integer to represent the height of the cropped image
        w: a positive integer to represent the width of the cropped image
    RETURNS:
        cropped_img: a 3d `np.ndarray` with dtype `np.uint8` created by copying the middle part of the given image
    '''
    num_rows, num_columns, num_dimensions = img.shape

    cropped_img = np.zeros((h, w, num_dimensions), dtype=np.uint8)

    start_height = max(0, (num_rows - h) // 2)
    start_width = max(0, (num_columns - w) // 2)
    
    for y in range(h):
        for x in range(w):
            cropped_img[y, x] = img[start_height + y, start_width + x]

    return cropped_img

#### Code Execution Times

In [6]:
img = load('../data/lse.jpg')

In [7]:
start_crop = time.time() 

cropped_img = crop(img, 600, 1000)

end_crop = time.time() 
run_time_crop = end_crop - start_crop
print(f'The total runtime for the crop() function was {run_time_crop} seconds')

The total runtime for the crop() function was 0.0001609325408935547 seconds


In [8]:
start_crop_w_loop = time.time() 

cropped_w_loop_image = crop_w_loop(img, 600, 1000)

end_crop_w_loop = time.time() 
run_time_crop_w_loop = end_crop_w_loop - start_crop_w_loop
print(f'The total runtime for the crop_w_loop() function was {run_time_crop_w_loop} seconds')

The total runtime for the crop_w_loop() function was 0.5724561214447021 seconds


4. (8 marks) Write the function definition for the function `get_constituent_img_lengths()` below, which is used to find out the length of one side of each image to form a squared collage needs to be. The function takes 3 positive integers (`size`, `b` and `num_imgs`). `size` represents the height (and width) of a squared collage to create later, `b` represents the width of each boundary (both inner and outer boundaries) of the collage to create later, and `num_imgs` is the number of images to have on one side of the collage image to create. The function returns a `list` of `int` of length `num_imgs`, with each `int` representing the length of one side of each image that needs to be cropped for the collage to have `num_imgs` on one side. Assume `size` is always not smaller than `b` $\times ($ `num_imgs` $+1) + $ `num_imgs`. 

    Note: sometimes it is not possible to have each length of one side of each image needs to be cropped to be exactly the same. For example, with `size = 10`, `b = 1` and `num_imgs = 2`, in order to have the lengths exactly the same for one side of the two images, the resulting width will be 3.5 which is not an `int`. Instead, the `list` to return can be `[3,4]` or `[4,3]`.

    Example usage:
        
    <img src="../figs/demo_3.png" width=600/>

    Some explanation:
    * With size 400, boundary size (b) 10 and the number of images on one side is 3, we have:
        * 10 (outer boundary), 120 (length of one side of the first image), 10 (inner boundary between image 1 and 2), 120 (length of one side of the second image), 10 (inner boundary between image 2 and 3), 120 (length of one side of the third image), 10 (outer boundary)
            * Note adding together we have 400
        * Therefore, the return value is `[120, 120, 120]`

In [9]:
def get_constituent_img_lengths(size, b, num_imgs):
    '''
    DESCRIPTION:
        this function returns a list of integers of length 'num_imgs', with each element of the list representing the length of one side of each image 
        that needs to be cropped for the collage to have 'num_img' on one side
    ----------
    PARAMETERS:
        size: a positive integer which represents the height (and width) of the collage
        b: a positive integer which represents the width of each boundary (both inner and outer boundaries) of the collage
        num_imgs: a positive integer which represents the number of images to have on each side of the collage
    RETURNS:
        list_img_sizes: a list of integers of length 'num_imgs', with each element of the list representing the length of one side of each image that 
                        needs to be cropped for the collage to have `num_imgs` on one side
    '''
    list_img_sizes = []

    actual_size = size - ((num_imgs + 1)*b)
    if actual_size // num_imgs == 0:   
        for _ in range(num_imgs):
            list_img_sizes.append(actual_size // num_imgs)
    else:
        for _ in range(num_imgs):
            list_img_sizes.append(actual_size // num_imgs)
        list_img_sizes[-1] = (actual_size // num_imgs) + 1

    assert (size > 0), \
    "size must be a positive integer, greater than 0"

    return list_img_sizes

5. (10 marks) Write the function definition for the function `create_collage_stack_horizontally()` to create a collage from a given set of images by combining them horizontally. The function takes 2 positive integers (`size` and `b`) and a `list` of images (`imgs`), with each image a 3d `np.ndarray` with dtype `np.uint8`. `size` represents the height (and width) of a squared collage to return, `b` represents the width of each boundary (both inner and outer boundaries) of the collage to create later and `imgs` represents the collection of constituent images to stack horizontally (from left to right). The function returns an image (3d `np.ndarray` with dtype `np.uint8`) with shape `(size, size, 3)` which represents a collage image formed by the given images stacked horizontally with boundary with `b`. Assume `size` is always not smaller than `b` $\times ($ `num_imgs` $+1) + $ `num_imgs`, and no image has a height less than `size` nor width less than the minimum values of the corresponding list created from `get_constituent_img_widths()`.

    Note: 
    * You may find your code for part 5 and part 6 quite similar. No need to worry about "copying and pasting code" for these parts
        * The similarity is due to the fact that part 5 is a "simplified" version of the original problem to be solved in part 6
    * You can have a small number of iterations (e.g. iterate over the list of given images), but no large number of iterations (e.g. iterate over each pixel of the given image) is needed
    * The images given as an argument should not be changed

    Example usage (suppose the images like `CKK.jpg` in the data folder have been loaded using an appropriate function from `image_util` to form variables like `ckk`, which are 3d `np.ndarray` representing images):

<img src="../figs/demo_5.png" width=600/>

In [53]:
def create_collage_stack_horizontally(size, b, imgs):
    '''
    DESCRIPTION:
        this function creates a collage from a given set of images by combining them horizontally
    ----------
    PARAMETERS:
        size: a positive integer which represents the height (and width) of a square collage (to be returned)
        b: a positive integer which represents the width of each boundary (both inner and outer boundaries) of the collage (to be returned)
        imgs: a list of images where each image is a 3d np.ndarray with dtype 
    RETURNS:
        collage: an image (3d np.ndarray with dtype np.uint8) with shape (size, size, 3) which represents a collage image formed by the given images 
                 stacked horizontally with boundary with 'b'
    '''
    collage = create_empty_square_img(size)
    num_rows, num_columns, num_dimensions = collage.shape   

    imgs_height = size - (2 * b)
    imgs_width = size - ((len(imgs) + 1) * b)

    cropped_imgs = []
    for img in imgs:
        cropped_imgs.append(crop(img, imgs_height, imgs_width))

    return collage


6. (10 marks) Write the function definition for the function `create_collage()` to create a collage from a given set of images. The function takes 4 positive integers (`size`, `b`, `nrow` and `ncol`) and a `list` of images (`imgs`), with each image a 3d `np.ndarray` with dtype `np.uint8`. `size` represents the height (and width) of a squared collage to return, `b` represents the width of each boundary (both inner and outer boundaries) of the resulted collage, `nrow` represents the number of images to have vertically in the resulted collage, `ncol` represents the number of images to have horizontally in the resulted collage and `imgs` represents the collection of constituent images. The function returns an image (3d `np.ndarray` with dtype `np.uint8`) with shape `(size, size, 3)` which represents a collage image formed by the given images with boundary with `b`. The collage is filled using the given images in the order of from left to right, then from top to bottom. Assume `size` is always not smaller than `b` x (`num_imgs`+1) + `num_imgs`, `nrow` x `ncol` equals the number of images in `imgs`, and no image has height and width less than the minimum values of the corresponding values from `get_constituent_img_lengths()`.

    Note: 
    * You may find your code for part 5 and part 6 quite similar. No need to worry about "copying and pasting code" for these parts
        * The similarity is due to the fact that part 5 is a "simplified" version of the original problem to be solved in part 6
    * You can have a small number of iterations (e.g. iterate over each row and columns), but no large number of iterations (e.g. iterate over each pixel of the given image) is needed
    * The images given as an argument should not be changed

    Example usage (suppose the images like `CKK.jpg` in the data folder have been loaded using an appropriate function from `image_util` to form variables like `ckk`, which are 3d `np.ndarray` representing images):

<img src="../figs/demo_6.png" width=600/>

In [59]:
def create_collage(size, b, nrow, ncol, imgs):
    '''
    DESCRIPTION:
        this function creates a collage from a given set of images
    ----------
    PARAMETERS:
        size: a positive integer which represents the height (and width) of a square collage (to be returned)
        b: a positive integer which represents the width of each boundary (both inner and outer boundaries) of the collage (to be returned)
        nrow: a positive integer which represents the number of images to have vertically in the resulted collage
        ncol: a positive integer which represents the number of images to have horizontally in the resulted collage
        imgs: a list of images where each image is a 3d np.ndarray with dtype 
    RETURNS:
        collage: an image (3d np.ndarray with dtype np.uint8) with shape (size, size, 3) which represents a collage image formed by the given images 
                 with boundary with 'b'
    '''
    num_horizontal_collages = len(imgs)/nrow
    num_imgs_in_row = len(imgs)/num_horizontal_collages

    collage = create_empty_square_img(size)
    num_rows, num_columns, num_dimensions = collage.shape

    print(num_imgs_in_row)

7. (3 marks) Demonstrate your function `create_collage()` works by using at least 6 given images to create a collage. You must show the created collage image below.

In [12]:
# The following code can be used to demonstrate the create_collage() function, and uses at leadt 6 images to create a collage. 
# However, due to our incompletion of the create_collage() function, the calling of the function will not show the created collage image below.

cbg = load('../data/CBG.jpg')
ckk = load('../data/CKK.jpg')
clm = load('../data/CLM.jpg')
col = load('../data/COL.jpg')
con = load('../data/CON_.jpg')
cow = load('../data/COW.jpg')

my_collage = create_collage(1000, 50, 3, 2, [cbg, ckk, clm, col, con, cow])
show(my_collage)

---

## Requirements

* You cannot use any modules and libraries (except `NumPy` and the given `image_util` in `src` folder) to answer this question
* This question is designed to assess your ability to manipulate `NumPy` array with _vectorised_ operations and indexing/slicing - therefore, please use `NumPy` and vectorise operations _wherever appropriate and possible_, unless it is stated otherwise in the question
* You can only use functionalities from `NumPy` that have been demonstrated in the lecture, and `np.full()` suggested above


---
## Testing and Assertion
The following code details the testing of the create_empty_square_img() function created in P1 of Q3.

In [20]:
test_cases = [
    (0, np.full((0, 0, 3), 200, dtype=np.uint8)), # boundary testing
    (1, np.array([[[200, 200, 200]]], dtype=np.uint8)), # minimum valid entry
    (10, np.full((10, 10, 3), 200, dtype=np.uint8)), # 'standard' input
    (20, np.full((20,20,3), 200, dtype=np.uint8)), # larger 'standard' input
    (100, np.full((100, 100, 3), 200, dtype=np.uint8)), # the example given in P1
    ] 

for size, expected_output in test_cases:
    actual_output = create_empty_square_img(size)
    # assert actual_output == expected_output, \
    assert np.array_equal(actual_output, expected_output), \
    f"Expected \"{expected_output}\" but returned \"{actual_output}\", argument: {size}."


Assertion takes place in P4 of Q3.